In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import laspy
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from scipy.spatial import cKDTree
from torch.utils.data import DataLoader, TensorDataset

In [ ]:

path_nic = "C:/Users/Nicola/Desktop/Jona" 

#  Alle .las-Dateien aus einem Ordner automatisch finden
def find_las_files(directory):
    return [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith(".las")]

#  Punktwolken aus allen Dateien einlesen und zusammenführen
def load_and_merge_las(files, chunk_size=100000):
    all_points = []
    for file in tqdm(files, desc="Lade .las-Dateien"):
        las = laspy.read(file)
        total_points = len(las.x)
        for start in range(0, total_points, chunk_size):
            end = min(start + chunk_size, total_points)
            chunk = np.vstack((
                las.x[start:end],
                las.y[start:end],
                las.z[start:end],
                # las.intensity[start:end],
                # las.return_number[start:end], las.number_of_returns[start:end],
                # las.classification[start:end]
            )).T
            all_points.append(chunk)
    return np.vstack(all_points)

#  Daten vorverarbeiten (Normalisierung)
def preprocess_data(points):
    scaler = StandardScaler()
    return scaler.fit_transform(points)

#  KD-Tree für Nachbarschaftssuche erstellen (optional)
def build_kdtree(points):
    return cKDTree(points)

#  PyTorch DataLoader erstellen
def create_dataloader(points, batch_size=64):
    tensor_data = torch.tensor(points, dtype=torch.float32)
    dataset = TensorDataset(tensor_data)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Hauptprogramm

directory = path_nic
las_files = find_las_files(directory)
if not las_files:
    print("Keine .las-Dateien gefunden!")
else:
    print(f"{len(las_files)} .las-Dateien gefunden.")
    points = load_and_merge_las(las_files)
    points_scaled = preprocess_data(points)


    print(f"Gesamtpunkte: {points.shape}")

